In [ ]:
// Nested tuples, the representation of abstract syntax trees; loaded once per session.
#load "../Support/Nested.fsx"

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"

# Merging States

The parsers generated by `Larkita` are *canonical* `LR(1)` parsers.  Canonical `LR(1)` automata often have many
more states than `LALR(1)` automata, because they keep apart states that only differ in the look-ahead sets of
their marked rules.  Such states have the same *core*.  With the argument `mergeStates = true`, `Larkita`
merges states with the same core, as long as this is safe:

1. Merging must not cause a new conflict, neither in the merged states nor in their successors.
2. Merging must not change how the contextual scanner splits the input into tokens.

Then the resulting parser accepts the same strings and returns the same abstract syntax trees as the canonical
parser.

## A Grammar for Arithmetic Expressions

We use the grammar from the notebook [02-Conflicts-Resolved.ipynb](02-Conflicts-Resolved.ipynb).

In [ ]:
#load "../Support/Larkita/Larkita.fsx"
open Larkita
open Nested

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "conflicts-resolved.lark")

We generate a canonical parser and a parser with merged states and compare the numbers of their states.

In [ ]:
let fileName = "conflicts-resolved.lark"
let canonical = Larkita.Open(fileName, outFile = "arith.out")
let merged    = Larkita.Open(fileName, outFile = "arith-merged.out", mergeStates = true)
printfn "canonical: %d states" (numberOfStates canonical)
printfn "merged:    %d states" (numberOfStates merged)
checkNotebook (numberOfStates canonical = 36 && numberOfStates merged = 19)

Both parsers return the same abstract syntax trees.

In [ ]:
for s in [ "1+2*3^4"; "2^3^4*5+6-7/8^9"; "(1+2)*3" ] do
    checkNotebook (parse canonical s = parse merged s)
    printfn "%s" (toString (parse merged s))

The output file of a parser with merged states contains the section `Merged states`.  It explains which states
with the same core have been kept apart.  The function `show_merged_states` prints this section.

In [ ]:
/// the part of text between the first occurrences of start and stop, like text[text.index(start):text.index(stop)]
let between (text: string) (start: string) (stop: string) =
    let i = text.IndexOf start
    text.Substring(i, text.IndexOf stop - i)

/// the states of an output file, like report.split('\n\n\nState ')[1:]
let states (report: string) = report.Split("\n\n\nState ") |> Array.skip 1 |> List.ofArray

In [ ]:
let showMergedStates (outFile: string) =
    let report = System.IO.File.ReadAllText outFile
    printfn "%s" ((between report "Merged states" "\nStates\n======").Trim())

In [ ]:
showMergedStates "arith-merged.out"

For this grammar, all states with the same core have been merged.  Hence, the merged automaton is the `LALR(1)`
automaton.  Every merged state lists the states of the canonical automaton that it has been merged from, and the
look-ahead set of every marked rule is the union of the look-ahead sets of this rule in these states.  The function
`show_state` prints the states that contain a given marked rule.

In [ ]:
let showState (outFile: string) (markedRule: string) =
    let report = System.IO.File.ReadAllText outFile
    for state in states report do
        if state.Contains markedRule then printfn "State %s\n" (state.Trim())

In [ ]:
showState "arith-merged.out" "atom → NUMBER •"

In the canonical automaton, the marked rule `atom → NUMBER •` occurs in two states:  The look-ahead set of the first
state contains `$END`, while the look-ahead set of the second state contains `")"`, because the second state is
reached inside parentheses.

In [ ]:
showState "arith.out" "atom → NUMBER •"

## The Grammar with the Mysterious Conflict

The grammar from the notebook [05-Mysterious-Conflicts.ipynb](05-Mysterious-Conflicts.ipynb) is an `LR(1)`
grammar, but it is not an `LALR(1)` grammar:  Merging the two states that contain the marked rule `a → X •`
would cause two reduce/reduce conflicts.  Therefore, `Larkita` keeps these states apart.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "mysterious.lark")

In [ ]:
let parser = Larkita.Open("mysterious.lark", outFile = "mysterious-merged.out", mergeStates = true)
showMergedStates "mysterious-merged.out"

All four strings of the language are still parsed correctly.

In [ ]:
for s in [ "vxy"; "vxz"; "wxy"; "wxz" ] do
    printfn "%s %s" s (toString (parse parser s))